# Parsing

In [ ]:
pip install scapy

In [ ]:
import os

#list EVERY dataset folder path needed to pull capture files from
#if everything is in 1 dataset, just list that 1 path

#COPY YOUR PATH INTO THE LIST AS SHOWN IN EXAMPLE
dataset_paths = [
#    "/datasets/pcaps-data/VPN-PCAPS-01",
#    "/datasets/pcaps-data/VPN-PCAPs-02",
#

]


capture_files = []

for dataset_path in dataset_paths:
    for root, dirs, files in os.walk(dataset_path):
        for f in files:
            if f.endswith(".pcap") or f.endswith(".pcapng"):
                capture_files.append(os.path.join(root, f))

print(f"\ntotal capture files found: {len(capture_files)}")

In [ ]:
#check for duplicate files
from collections import Counter

filenames_only = [os.path.basename(f) for f in capture_files]
name_counts = Counter(filenames_only)

duplicates = {name: count for name, count in name_counts.items() if count > 1}
print(f"duplicate filenames found: {len(duplicates)}")
if duplicates:
    print(duplicates)

In [ ]:
#functions to get the label and stream id from a filename
#and the PCAP parsing function
import re
from scapy.all import PcapReader, IP, TCP, UDP

def get_stream_id(filename):
    name = filename.replace(".pcapng", "").replace(".pcap", "") #pcapng first if not it wont be replaced correctly
    match = re.match(r"^(.*)([a-zA-Z])$", name)
    if match and re.search(r"\d[a-zA-Z]$", name):
        stream_id = match.group(1)
        part = match.group(2)
    else:
        stream_id = name
        part = None
    return stream_id, part

#reads filename and assigns network and category
def get_label(filename, network_type):
    name = filename.lower()

    #activity type
    #video here means LIVE video
    #streaming means the regular non live video
    categories = {
        "chat": ["chat"],
        "email": ["email", "gmail", "mail", "thunderbird", "imap", "pop"],
        "audio": ["audio", "voip", "voipbuster", "voice"],
        "video": ["video"], 
        "file_transfer": ["ftp", "ftps", "sftp", "scp", "skype_file", "skypefile", "filetransfer", "skype_transfer"],
        "p2p": ["torrent", "bittorrent", "p2p"],
        "streaming": ["spotify", "youtube", "netflix", "vimeo"],
        "browsing": ["browsing", "facebook", "google", "twitter", "ssl"]
    }

    category = "unknown"
    for cat, keywords in categories.items():
        if any(k in name for k in keywords):
            category = cat
            break

    return network_type, category
    
#PCAP parsing function
def parse_pcap_to_flows(filepath):
    flows = {}
    with PcapReader(filepath) as pcap_reader:
        for pkt in pcap_reader:
            if IP not in pkt:
                continue
            if TCP in pkt:
                proto = "TCP"
                sport = pkt[TCP].sport
                dport = pkt[TCP].dport
            elif UDP in pkt:
                proto = "UDP"
                sport = pkt[UDP].sport
                dport = pkt[UDP].dport
            else:
                continue

            src_ip = pkt[IP].src
            dst_ip = pkt[IP].dst

            if (src_ip, sport) < (dst_ip, dport):
                flow_key = (src_ip, sport, dst_ip, dport, proto)
                direction = 1
            else:
                flow_key = (dst_ip, dport, src_ip, sport, proto)
                direction = -1

            pkt_size = len(pkt)
            pkt_time = float(pkt.time)

            if flow_key not in flows:
                flows[flow_key] = []
            flows[flow_key].append((pkt_time, pkt_size, direction))

    return flows

In [ ]:
import os
import gc
import pickle

os.makedirs("parsed_flows", exist_ok=True)

#CHECK B4 RUNNING
NETWORK_TYPE = "" 

processed_count = 0
skipped_count = 0
failed_count = 0

for filepath in capture_files:
    filename = os.path.basename(filepath)
    safe_name = filename.replace(".", "_")
    out_path = f"parsed_flows/{safe_name}.pkl"

    if os.path.exists(out_path):
        skipped_count += 1
        continue

    stream_id, part = get_stream_id(filename)
    network_type, category = get_label(filename, network_type=NETWORK_TYPE)

    try:
        file_flows = parse_pcap_to_flows(filepath)
    except Exception as e:
        print(f"failed on {filename}: {e}")
        failed_count += 1
        continue

    file_records = []
    for flow_key, packet_seq in file_flows.items():
        file_records.append({
            "filepath": filepath,
            "filename": filename,
            "stream_id": stream_id,
            "part": part,
            "network_type": network_type,
            "category": category,
            "flow_key": flow_key,
            "packet_seq": packet_seq
        })

    with open(out_path, "wb") as f:
        pickle.dump(file_records, f)

    print(f"done: {filename} -> {len(file_records)} flows")
    processed_count += 1

    del file_flows, file_records
    gc.collect()

print(f"\nbatch complete: {NETWORK_TYPE}")
print(f"newly processed: {processed_count}")
print(f"skipped (already done): {skipped_count}")
print(f"failed: {failed_count}")
print(f"total files in capture_files: {len(capture_files)}")

In [ ]:
#see which files labelled as unknown
unknown_files = df[df["category"] == "unknown"]["filename"].unique()
print(unknown_files)

In [ ]:
import glob
import pandas as pd

#only combine the pickles belonging to files from THIS batch's capture_files
#matches by filename, so it only picks up what was just parsed in this run
batch_filenames = set(os.path.basename(f) for f in capture_files)

batch_records = []
for pkl_path in glob.glob("parsed_flows/*.pkl"):
    with open(pkl_path, "rb") as f:
        file_records = pickle.load(f)
        if file_records and file_records[0]["filename"] in batch_filenames:
            batch_records.extend(file_records)

df = pd.DataFrame(batch_records)
print(f"total flows in this batch ({NETWORK_TYPE}): {len(df)}")
print(df["category"].value_counts())

#save this batch's dataframe with a name based on its network type
df.to_pickle(f"nonvpn2_files_AL2.pkl")

In [ ]:
first_row = df.iloc[0]
for col in df.columns:
    if col == "packet_seq":
        print(f"{col}: {first_row[col][:5]} ... ({len(first_row[col])} total packets)")
    else:
        print(f"{col}: {first_row[col]}")